In [0]:
orders_df = spark.read.format("delta").load("/Volumes/pei/silver/orders/")

In [0]:
customers_df = spark.read.format("delta").load("/Volumes/pei/silver/customers/")

In [0]:
products_df = spark.read.format("delta").load("/Volumes/pei/silver/products/")

In [0]:
from pyspark.sql.functions import round, col

fact_orders = (
    orders_df
    .join(customers_df, "customer_id", "inner")
    .join(products_df, "product_id", "inner")
    .withColumn("profit", round(col("profit"), 2))
    # .select(
    #     "order_id",
    #     "order_date",
    #     "customer_id",
    #     "product_id",
    #     "category",
    #     "subcategory",
    #     "sales",
    #     "profit"
    )

In [0]:
display(fact_orders)

In [0]:
from pyspark.sql.functions import year, sum

agg_profit = (
    fact_orders
    .withColumn("year", year(col("order_date")))
    .groupBy(
        "year",
        "category",
        "subcategory",
        "customer_Id"
    )
    .agg(sum("profit").alias("total_profit"))
)
display(agg_profit)

In [0]:
fact_orders.createOrReplaceTempView("fact_orders")

In [0]:
%sql
SELECT year(order_date) AS year, SUM(profit) AS total_profit
FROM fact_orders
GROUP BY year

In [0]:
%sql
SELECT year(order_date), category, ROUND(SUM(profit),2)
FROM fact_orders
GROUP BY year(order_date), category

In [0]:
%sql
SELECT customer_id, ROUND(SUM(profit),2) AS profit_per_customer
FROM fact_orders
GROUP BY customer_id

In [0]:
%sql
SELECT  year(order_date) as year, customer_id, round(SUM(profit),2) as profit_per_year_per_customer
FROM fact_orders
GROUP BY year(order_date),  customer_id